# COVID · OpenKG 纯时序版：只做“数据允许做的事”

这个 Notebook 不使用 LLM。它读取 OpenKG 的 `event-covid-19-v0.2.json`（JSON-LD），
把新闻事件（`P1`）和政策事件（`P6`）写成 `COVIDEvent` 节点，再按 `P5` / `P16` 字段连 `FOLLOWED_BY`。

**为什么不用 LLM 补因果？**

因为这份数据本身**没有因果标注**。在 194 条事件上让通用模型“凭空推理因果”，
我们在 V2/V3 里已经看到后果：时间倒置、模板化原因、高频实体成为辐射中心。
所以这个 Notebook 的工程决策是：**只保留数据明确声明的先后关系。**

> 这不是“放弃”，而是“知道边界”。

In [ ]:
# ============================================================
# 0. 环境与路径
# ============================================================
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
RESET_DB = os.getenv("RESET_DB", "false").lower() == "true"

# 默认保留 OpenKG 原标注；设为 true 时只写入通过时间戳校验的边
STRICT_TIME_ORDER = os.getenv("STRICT_TIME_ORDER", "false").lower() == "true"

if not NEO4J_PASSWORD:
    raise RuntimeError("缺少 NEO4J_PASSWORD：请复制 .env.example 为 .env 并填写。")

DATA_PATH = PROJECT_ROOT / "data" / "event-covid-19-v0.2.json"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"找不到数据文件：{DATA_PATH}\n"
        "请按 data/README.md 的说明下载 event-covid-19-v0.2.json，"
        "放到 data/ 目录下。本仓库不重新分发该数据。"
    )

print("数据文件:", DATA_PATH)
print("Neo4j:", NEO4J_URI, "| 用户:", NEO4J_USER)
print("RESET_DB:", RESET_DB, "| STRICT_TIME_ORDER:", STRICT_TIME_ORDER)


## 1. 解析 JSON-LD

字段映射：

| 字段 | 含义 | 用途 |
|---|---|---|
| `@id` | 事件 URI | 节点主键 |
| `P1` | 新闻标题 | 新闻事件标题 |
| `P0` | 新闻时间 | 新闻事件时间 |
| `P2` | 新闻摘要 | 节点属性 |
| `P4` | 新闻来源 | 节点属性 |
| `P5` | 之前发生的事件 | 时序边 |
| `P6` | 政策标题 | 政策事件标题 |
| `P7` | 政策发布日期 | 政策事件时间 |
| `P9` | 发布机构 | 节点属性 |
| `P12` / `P15` | 政策摘要/正文 | 节点属性 |
| `P16` | 数据集中的前序政策指针 | 时序边 |

> 注意：`P16` 的中文标签是“后续政策”，但在本数据集中，它的实际用法与 `P5` 一样，
> 是指向**链上的前一个节点**。代码不盲信字段名，而是用时间戳做二次校验。

In [ ]:
import json
import re
from collections import Counter
from datetime import datetime

data = json.loads(DATA_PATH.read_text(encoding="utf-8"))
graph = data.get("@graph", [])
if not isinstance(graph, list):
    raise RuntimeError("JSON-LD 结构异常：@graph 不是数组。")


def short_id(uri):
    """把完整 URI 压成 R53 这样的短 ID；只用于展示和兼容。"""
    text = str(uri or "").strip().rstrip("/")
    return text.split("/")[-1] if text else ""


def parse_time(raw):
    """兼容新闻的 02/05/2020 格式和政策的 2020年02月11日 格式。"""
    text = str(raw or "").strip()
    if not text:
        return None
    for fmt in ("%m/%d/%Y, %H:%M:%S", "%Y年%m月%d日", "%Y-%m-%d"):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            continue
    return None


events = []
for item in graph:
    if not isinstance(item, dict):
        continue

    if "P1" in item:
        kind = "news"
        title = str(item.get("P1") or "").strip()
        time_raw = item.get("P0") or ""
        content = str(item.get("P2") or "")
        source = str(item.get("P4") or "")
        link_raw = item.get("P5") or ""
        link_source = "openkg_P5"
    elif "P6" in item:
        kind = "policy"
        title = str(item.get("P6") or "").strip()
        time_raw = item.get("P7") or ""
        content = str(item.get("P12") or item.get("P15") or "")
        source = str(item.get("P9") or "")
        link_raw = item.get("P16") or ""
        link_source = "openkg_P16"
    else:
        continue

    event_id = str(item.get("@id") or "").strip()
    if not event_id or not title:
        continue

    events.append({
        "id": event_id,
        "short_id": short_id(event_id),
        "kind": kind,
        "title": title,
        "time_raw": str(time_raw),
        "dt": parse_time(time_raw),
        "content": content[:1000],
        "source": source,
        "link_raw": str(link_raw),
        "link_source": link_source,
        "link_id": None,
    })

if not events:
    raise RuntimeError("没有解析到任何事件。")

kind_counts = Counter(ev["kind"] for ev in events)
print(f"解析到 {len(events)} 个事件：新闻 {kind_counts['news']} / 政策 {kind_counts['policy']}")


## 2. 解析 `P5` / `P16` 指针：`P16` 是“后续”，不是“前序”

这里做三件事：

1. 用完整 URI 建索引，同时保留短 ID 索引，兼容“长 ID / 短 ID”两种写法；
2. 解析每条指针，统计无法解析的边；
3. **按字段语义决定边方向，再用时间戳校验**。

字段语义（本次数据实测）：

| 字段 | 标签 | 55/140 条中的方向 | 正确边方向 |
|---|---|---|---|
| `P5` | 之前发生 | 绝大多数指向更早或同日；1 条同天逆序 | `target -> current` |
| `P16` | 后续政策 | 119 条同日，20 条更晚，1 条缺失 | `current -> target` |

> 原始实现把 `P16` 也当成“前序指针”，统一写成 `target -> current`。
> 结果：139 条 `P16` 全部反向（20 条跨天 + 119 条同日），
> 再加上 1 条 `P5` 同天逆序、1 条 `P16` 跨类，共 22 条边在时间上倒置。
> 这正是“字段名不一定是语义，时间戳也不一定能暴露方向错误”的典型例子。

本 Notebook 的处理：

- `P5`：`target -> current`；
- `P16`：`current -> target`；
- 同时计算 `time_valid`：两端都有时间且 `start <= end` 才为 `true`；
- 缺时间的边保留，但标记 `time_valid=false`。

In [ ]:
by_full = {ev["id"]: ev for ev in events}
by_short = {ev["short_id"]: ev for ev in events}

if len(by_full) != len(events):
    raise RuntimeError("存在重复的 @id，请先检查数据。")
if len(by_short) != len(events):
    raise RuntimeError("存在重复的短 ID，不能安全使用短 ID 索引。")

unresolved = []
for ev in events:
    if not ev["link_raw"]:
        continue
    target = by_full.get(ev["link_raw"]) or by_short.get(short_id(ev["link_raw"]))
    if target is None:
        unresolved.append((ev["short_id"], ev["link_raw"]))
    else:
        ev["link_id"] = target["id"]


edges = []
for ev in events:
    if not ev["link_id"]:
        continue
    target = by_full[ev["link_id"]]

    # 关键：P5 是“之前”，P16 是“后续”，方向不能共用同一套写法
    if ev["link_source"] == "openkg_P5":
        start, end = target, ev
    else:  # openkg_P16
        start, end = ev, target

    time_valid = (
        start["dt"] is not None
        and end["dt"] is not None
        and start["dt"] <= end["dt"]
    )
    edges.append({
        "start_id": start["id"],
        "end_id": end["id"],
        "source": ev["link_source"],
        "time_valid": time_valid,
        "start_short": start["short_id"],
        "end_short": end["short_id"],
        "start_time": start["time_raw"],
        "end_time": end["time_raw"],
    })

# 理论上每个指针一条边；如果出现重复，先去重再建图
seen = set()
unique_edges = []
for edge in edges:
    key = (edge["start_id"], edge["end_id"])
    if key in seen:
        continue
    seen.add(key)
    unique_edges.append(edge)
if len(unique_edges) != len(edges):
    print(f"发现 {len(edges) - len(unique_edges)} 条重复边，已去重。")
edges = unique_edges


def edge_issue(edge):
    start = by_full[edge["start_id"]]
    end = by_full[edge["end_id"]]
    if start["dt"] is None or end["dt"] is None:
        return "缺时间"
    if start["dt"] > end["dt"]:
        return "时间倒置"
    return None


issues = [(edge, edge_issue(edge)) for edge in edges]
issues = [(edge, reason) for edge, reason in issues if reason]
inversions = [(edge, reason) for edge, reason in issues if reason == "时间倒置"]
unknown_time = [(edge, reason) for edge, reason in issues if reason == "缺时间"]

print(f"可解析的 P5/P16 指针：{len(edges)} 条")
print(f"无法解析的指针：{len(unresolved)} 条")
for row in unresolved[:5]:
    print("  ", row)
print(f"时间倒置：{len(inversions)} 条；缺时间：{len(unknown_time)} 条")
for edge, reason in issues[:5]:
    print(
        f"  [{reason}] {edge['start_short']} ({edge['start_time']})"
        f" -> {edge['end_short']} ({edge['end_time']})"
    )


## 3. 连通分量：这张图到底碎不碎？

把 `FOLLOWED_BY` 当成无向边做并查集，统计连通分量。
如果分量数远大于 1，说明数据是“多条独立链”，不是一张全局时间线。

In [ ]:
parent = {ev["id"]: ev["id"] for ev in events}


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb


for edge in edges:
    union(edge["start_id"], edge["end_id"])

components = Counter(find(ev["id"]) for ev in events)
print(f"连通分量：{len(components)} 个")
print("最大的 5 个分量：", components.most_common(5))


In [ ]:
# ============================================================
# 4. 连接 Neo4j
# ============================================================
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Neo4j 连接成功")


## 4. 建图：`COVIDEvent` + `FOLLOWED_BY`

默认保留全部 194 条指针；`STRICT_TIME_ORDER=true` 时只写入时间可验证的 192 条
（丢弃 2 条时间倒置）。
每条边都带 `source` 属性，标明它来自 `P5` 还是 `P16`，方便追溯。

In [ ]:
def ensure_constraints(tx):
    tx.run("""
        CREATE CONSTRAINT covid_event_id_unique IF NOT EXISTS
        FOR (e:COVIDEvent) REQUIRE e.id IS UNIQUE
    """)


def reset_database(tx):
    tx.run("MATCH (n) DETACH DELETE n")


def build_covid_graph(tx, ev_list, edge_list):
    for ev in ev_list:
        tx.run("""
            MERGE (e:COVIDEvent {id: $id})
            SET e.short_id = $short_id,
                e.title = $title,
                e.time = $time,
                e.timestamp = $timestamp,
                e.content = $content,
                e.source = $source,
                e.kind = $kind,
                e.dataset = 'OpenKG event-covid-19-v0.2'
        """, id=ev["id"], short_id=ev["short_id"], title=ev["title"],
             time=ev["time_raw"],
             timestamp=int(ev["dt"].timestamp()) if ev["dt"] else None,
             content=ev["content"], source=ev["source"], kind=ev["kind"])

    for edge in edge_list:
        tx.run("""
            MATCH (start:COVIDEvent {id: $start_id})
            MATCH (end:COVIDEvent {id: $end_id})
            MERGE (start)-[r:FOLLOWED_BY]->(end)
            SET r.source = $source,
                r.declared_order = true,
                r.time_valid = $time_valid
        """, start_id=edge["start_id"], end_id=edge["end_id"],
             source=edge["source"], time_valid=edge["time_valid"])


edge_list = [e for e in edges if (not STRICT_TIME_ORDER or e["time_valid"])]

with driver.session() as session:
    session.execute_write(ensure_constraints)
    if RESET_DB:
        session.execute_write(reset_database)
        print("已清空数据库（RESET_DB=true）")
    session.execute_write(build_covid_graph, events, edge_list)

print(f"写入 {len(events)} 个节点，{len(edge_list)} 条 FOLLOWED_BY（严格模式={STRICT_TIME_ORDER}）")


In [ ]:
# ============================================================
# 5. 验收
# ============================================================
with driver.session() as session:
    node_count = session.run(
        "MATCH (e:COVIDEvent) RETURN count(e) AS c"
    ).single()["c"]
    edge_count = session.run(
        "MATCH (:COVIDEvent)-[r:FOLLOWED_BY]->(:COVIDEvent) RETURN count(r) AS c"
    ).single()["c"]
    source_rows = session.run("""
        MATCH (:COVIDEvent)-[r:FOLLOWED_BY]->(:COVIDEvent)
        RETURN r.source AS source, count(*) AS c
        ORDER BY c DESC
    """).data()

print(f"COVIDEvent 节点：{node_count}")
print(f"FOLLOWED_BY 边：{edge_count}")
print("按来源分布：")
for row in source_rows:
    print(f"  {row['source']}: {row['c']}")


In [ ]:
# ============================================================
# 6. Neo4j Browser 查询
# ============================================================
queries = [
    "MATCH p=(:COVIDEvent)-[:FOLLOWED_BY]->(:COVIDEvent) RETURN p LIMIT 50;",
    "MATCH (e:COVIDEvent) RETURN e.title, e.time ORDER BY e.timestamp DESC LIMIT 20;",
    "MATCH (:COVIDEvent)-[r:FOLLOWED_BY]->(:COVIDEvent) WHERE r.time_valid = false RETURN count(r);",
]
for q in queries:
    print(q)


In [ ]:
driver.close()
print("Neo4j 连接已关闭")


## 6. 结论：为什么停在“时序”

- `FOLLOWED_BY` 来自 OpenKG 的显式指针，**这是数据允许我们确定的关系**；
- 但“显式指针”不等于“方向可以直接用”：`P16` 是后续指针，旧实现把它当前序，制造了 22 条反向边；
- 按字段语义定向后，仍有 2 条时间倒置（1 条 P5 同天逆序、1 条 P16 跨类指向 1 月新闻），说明“声明顺序”不等于“严格时间”；
- 因此本 Notebook 同时保留 `declared_order` 和 `time_valid` 两个属性，不把任何一条边伪装成“绝对正确”；
- 如果业务要求严格时间线，设 `STRICT_TIME_ORDER=true`，只写入通过校验的 192 条边。

**这条边界比多连 100 条因果线更有价值。**